# Silver configuration mappings

## Purpose

Create the five small, versioned configuration tables that Silver transformations will use. These tables separate approved business decisions from transformation code. This milestone does not clean Bronze, build the remaining Silver outputs, or create Gold tables.

## Safe reruns

`CREATE TABLE IF NOT EXISTS` preserves reviewed rows. Seeded rules use a deterministic `MERGE`, so rerunning the notebook does not append duplicates. Coverage queries read Bronze but never write to it.

## 1. Place-name aliases

Grain: one context-specific source alias per alias version. Region, province, municipality, and place type are part of the natural key so a common name such as `Poblacion` is never treated as a global match. The DPWH region aliases from issue #87 are seeded as version `dpwh-region-aliases-2026-10-v1`. Each of the 17 rows maps one DPWH region spelling to exactly one official PSGC region. Decision D-41 approves them, so the project-region map uses them in its approved-alias stage. `Region IV-B` maps to `MIMAROPA Region`: its 700 projects are `For Procurement` work from MIMAROPA district offices with no budget yet, and none repeats a `MIMAROPA Region` contract. District engineering office names, `2026`, and blank regions are not region names and stay unresolved. The Census Table C aliases from issue #88 are seeded in the next cell as version `census-table-c-aliases-2026-10-v1`. Each is backed by PSGC evidence from that issue or from the review of the rows still unmatched after the first run. Two map a shortened sheet name to its PSGC place and must name the target region. Two map a renamed municipality heading to its PSGC municipality and must name the province. Fifteen map a barangay spelled differently from PSGC and must name its city or municipality. `match_rule` records how the population notebook uses each alias: `TABLE_C_SHEET_NAME` for a sheet, `TABLE_C_LOCALITY_HEADING` for a heading. Every other approved Table C alias applies to a single source row.

In [0]:
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `02-silver`;

CREATE TABLE IF NOT EXISTS `02-silver`.config_place_name_alias (
    source_system STRING NOT NULL,
    raw_place_name STRING NOT NULL,
    raw_region_name STRING NOT NULL,
    raw_province_name STRING NOT NULL,
    raw_city_municipality_name STRING NOT NULL,
    place_type STRING NOT NULL,
    canonical_psgc_code STRING,
    canonical_place_name STRING,
    match_rule STRING,
    alias_version STRING NOT NULL,
    psgc_version STRING,
    approval_status STRING NOT NULL,
    approved_by STRING,
    approved_at TIMESTAMP,
    is_active BOOLEAN NOT NULL,
    source_reference STRING,
    notes STRING
)
USING DELTA
COMMENT 'Context-specific, reviewed source place aliases for Silver matching';

MERGE INTO `02-silver`.config_place_name_alias AS target
USING (
    SELECT
        'dpwh_projects' AS source_system,
        raw_place_name,
        '' AS raw_region_name,
        '' AS raw_province_name,
        '' AS raw_city_municipality_name,
        'REGION' AS place_type,
        canonical_psgc_code,
        canonical_place_name,
        'Exact DPWH reported_region spelling to one official PSGC region' AS match_rule,
        'dpwh-region-aliases-2026-10-v1' AS alias_version,
        '2Q 2026 as of 2026-06-30' AS psgc_version,
        'APPROVED' AS approval_status,
        'Buildabida' AS approved_by,
        TIMESTAMP '2026-10-09 00:00:00' AS approved_at,
        TRUE AS is_active,
        'Issue #87 approved decision (D-41), DPWH region coverage profile and Silver PSGC region list' AS source_reference,
        'Region IV-B rows are 700 For Procurement projects from MIMAROPA DEOs with no budget yet. They are not duplicates of MIMAROPA Region rows.' AS notes
    FROM VALUES
        ('Region I', '0100000000', 'Region I (Ilocos Region)'),
        ('Region II', '0200000000', 'Region II (Cagayan Valley)'),
        ('Region III', '0300000000', 'Region III (Central Luzon)'),
        ('Region IV-A', '0400000000', 'Region IV-A (CALABARZON)'),
        ('Region V', '0500000000', 'Region V (Bicol Region)'),
        ('Region VI', '0600000000', 'Region VI (Western Visayas)'),
        ('Region VII', '0700000000', 'Region VII (Central Visayas)'),
        ('Region VIII', '0800000000', 'Region VIII (Eastern Visayas)'),
        ('Region IX', '0900000000', 'Region IX (Zamboanga Peninsula)'),
        ('Region X', '1000000000', 'Region X (Northern Mindanao)'),
        ('Region XI', '1100000000', 'Region XI (Davao Region)'),
        ('Region XII', '1200000000', 'Region XII (SOCCSKSARGEN)'),
        ('National Capital Region', '1300000000', 'National Capital Region (NCR)'),
        ('Cordillera Administrative Region', '1400000000', 'Cordillera Administrative Region (CAR)'),
        ('Region XIII', '1600000000', 'Region XIII (Caraga)'),
        ('Negros Island Region', '1800000000', 'Negros Island Region (NIR)'),
        ('Region IV-B', '1700000000', 'MIMAROPA Region')
    AS seed(raw_place_name, canonical_psgc_code, canonical_place_name)
) AS source
ON target.source_system = source.source_system
AND target.raw_place_name = source.raw_place_name
AND target.raw_region_name = source.raw_region_name
AND target.raw_province_name = source.raw_province_name
AND target.raw_city_municipality_name = source.raw_city_municipality_name
AND target.place_type = source.place_type
AND target.alias_version = source.alias_version
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
MERGE INTO `02-silver`.config_place_name_alias AS target
USING (
    SELECT
        'census_2024_table_c' AS source_system,
        raw_place_name,
        raw_region_name,
        raw_province_name,
        raw_city_municipality_name,
        place_type,
        canonical_psgc_code,
        canonical_place_name,
        match_rule,
        'census-table-c-aliases-2026-10-v1' AS alias_version,
        CAST(NULL AS STRING) AS psgc_version,
        'APPROVED' AS approval_status,
        'Buildabida' AS approved_by,
        TIMESTAMP '2026-10-09 00:00:00' AS approved_at,
        TRUE AS is_active,
        'Issue #88 evidence queries and the 2026-10-09 review of unmatched Table C rows' AS source_reference,
        notes
    FROM VALUES
        ('City of Lapu', 'Region VII (Central Visayas)', '', '', 'CITY', '0731100000', 'CITY OF LAPU-LAPU', 'TABLE_C_SHEET_NAME',
            'Table C shortens the sheet name. All 30 barangays on the sheet match inside 07311.'),
        ('SGA', 'Bangsamoro Autonomous Region In Muslim Mindanao (BARMM)', '', '', 'UNASSIGNED', '1999900000', 'SPECIAL GEOGRAPHIC AREA', 'TABLE_C_SHEET_NAME',
            'Table C abbreviates the BARMM Special Geographic Area. Its barangays match inside 19999.'),
        ('SAN ISIDRO', '', 'Davao del Norte', '', 'MUNICIPALITY', '1102324000', 'SAWATA', 'TABLE_C_LOCALITY_HEADING',
            'Table C uses the former name. Barangays listed under it, such as KIPALILI, are in SAWATA.'),
        ('DON VICTORIANO CHIONGBIAN', '', 'Misamis Occidental', '', 'MUNICIPALITY', '1004217000', 'DON VICTORIANO', 'TABLE_C_LOCALITY_HEADING',
            'Table C uses the full name. Barangays listed under it, such as GANDAWAN, are in DON VICTORIANO.'),
        ('Lurogan', '', 'Bukidnon', 'City of Valencia', 'BARANGAY', '1001321017', 'LURUGAN', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 1 letter. Reviewed against the barangays of this locality.'),
        ('Kahapunan', '', 'Bukidnon', 'City of Valencia', 'BARANGAY', '1001321011', 'KAHAPONAN', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 1 letter. Reviewed against the barangays of this locality.'),
        ('Merangerang', '', 'Bukidnon', 'Quezon', 'BARANGAY', '1001317019', 'MERANGERAN', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 1 letter. Reviewed against the barangays of this locality.'),
        ('Ambuclao', '', 'Benguet', 'Bokod', 'BARANGAY', '1401104001', 'AMBUKLAO', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 1 letter. Reviewed against the barangays of this locality.'),
        ('Indalaza', '', 'Bukidnon', 'City of Malaybalay', 'BARANGAY', '1001312018', 'INDALASA', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 1 letter. Reviewed against the barangays of this locality.'),
        ('Villa Floresca', '', 'Nueva Ecija', 'San Jose City', 'BARANGAY', '0304926040', 'VILLA FLORESTA', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 1 letter. Reviewed against the barangays of this locality.'),
        ('Daclan', '', 'Benguet', 'Bokod', 'BARANGAY', '1401104004', 'DAKLAN', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 1 letter. Reviewed against the barangays of this locality.'),
        ('Balukbukan', '', 'Bukidnon', 'Kitaotao', 'BARANGAY', '1001309002', 'BALOCBOCAN', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 4 letters. Reviewed against the barangays of this locality.'),
        ('Baborawon', '', 'Bukidnon', 'Kalilangan', 'BARANGAY', '1001307002', 'BARORAWON', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 1 letter. Reviewed against the barangays of this locality.'),
        ('Kabalabag', '', 'Bukidnon', 'City of Malaybalay', 'BARANGAY', '1001312021', 'KIBALABAG', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 1 letter. Reviewed against the barangays of this locality.'),
        ('Lubusan', '', 'Zamboanga del Sur', 'Lapuyan', 'BARANGAY', '0907313012', 'LUBOSAN', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 1 letter. Reviewed against the barangays of this locality.'),
        ('Ginitligan', '', 'Catanduanes', 'Baras', 'BARANGAY', '0502002010', 'GENITLIGAN', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 1 letter. Reviewed against the barangays of this locality.'),
        ('Pinagsakahan', '', 'Quezon', 'Calauag', 'BARANGAY', '0405607061', 'PINAGSAKAYAN', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 1 letter. Reviewed against the barangays of this locality.'),
        ('Culasi', '', 'Bukidnon', 'Sumilao', 'BARANGAY', '1001319002', 'KULASI', 'TABLE_C_BARANGAY_SPELLING',
            'Spelling differs from PSGC by 1 letter. Reviewed against the barangays of this locality.'),
        ('Barangay ng mga Mangingisda', '', '', 'City of Puerto Princesa', 'BARANGAY', '1731500062', 'MANGINGISDA', 'TABLE_C_BARANGAY_NAME',
            'Table C writes the full name. PSGC lists the same barangay of Puerto Princesa as MANGINGISDA.')
    AS seed(
        raw_place_name, raw_region_name, raw_province_name, raw_city_municipality_name, place_type,
        canonical_psgc_code, canonical_place_name, match_rule, notes
    )
) AS source
ON target.source_system = source.source_system
AND target.raw_place_name = source.raw_place_name
AND target.raw_region_name = source.raw_region_name
AND target.raw_province_name = source.raw_province_name
AND target.raw_city_municipality_name = source.raw_city_municipality_name
AND target.place_type = source.place_type
AND target.alias_version = source.alias_version
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT_IF(approval_status = 'APPROVED' AND is_active) AS active_approved_rows,
    COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
        'source_system', source_system,
        'raw_place_name', raw_place_name,
        'raw_region_name', raw_region_name,
        'raw_province_name', raw_province_name,
        'raw_city_municipality_name', raw_city_municipality_name,
        'place_type', place_type,
        'alias_version', alias_version
    )) AS duplicate_natural_keys
FROM `02-silver`.config_place_name_alias;

## 2. Project-category mappings

Grain: one source category and infrastructure-type pair per taxonomy version. For the approved DPWH taxonomy, `raw_category` stays blank and `source_infra_type` stores one normalized atomic value extracted from `source_record_json.componentCategories`. The raw DPWH `category` field remains source evidence and is not the primary taxonomy input.

In [0]:
CREATE TABLE IF NOT EXISTS `02-silver`.config_project_category_mapping (
    source_system STRING NOT NULL,
    raw_category STRING NOT NULL,
    source_infra_type STRING NOT NULL,
    standardized_sector STRING,
    standardized_category STRING,
    is_flood_related BOOLEAN,
    mapping_rule STRING,
    taxonomy_version STRING NOT NULL,
    approval_status STRING NOT NULL,
    approved_by STRING,
    approved_at TIMESTAMP,
    is_active BOOLEAN NOT NULL,
    source_reference STRING,
    notes STRING
)
USING DELTA
COMMENT 'Reviewed project category and infrastructure type mappings by taxonomy version';

MERGE WITH SCHEMA EVOLUTION INTO `02-silver`.config_project_category_mapping AS target
USING (
    SELECT
        source_system,
        raw_category,
        source_infra_type,
        standardized_sector,
        standardized_category,
        is_flood_related,
        'Exact normalized componentCategories token' AS mapping_rule,
        'dpwh-component-categories-2026-10-v1' AS taxonomy_version,
        'APPROVED' AS approval_status,
        'Buildabida' AS approved_by,
        TIMESTAMP '2026-10-09 00:00:00' AS approved_at,
        TRUE AS is_active,
        'Issue #81 approved decision and selected DPWH source profile' AS source_reference,
        'The raw DPWH category remains source evidence. Multi-sector is derived after approved token mapping.' AS notes
    FROM VALUES
        ('dpwh_projects', '', 'Bridges', 'Bridges', 'Bridges', FALSE),
        ('dpwh_projects', '', 'Buildings and Facilities', 'Buildings and Facilities', 'Buildings and Facilities', FALSE),
        ('dpwh_projects', '', 'Consultancy', 'Consultancy', 'Consultancy', FALSE),
        ('dpwh_projects', '', 'Flood Control and Drainage', 'Flood Control and Drainage', 'Flood Control and Drainage', TRUE),
        ('dpwh_projects', '', 'Roads', 'Roads', 'Roads', FALSE),
        ('dpwh_projects', '', 'Septage and Sewerage Plants', 'Septage and Sewerage Plants', 'Septage and Sewerage Plants', FALSE),
        ('dpwh_projects', '', 'Water Provision and Storage', 'Water Provision and Storage', 'Water Provision and Storage', FALSE)
    AS seed(
        source_system, raw_category, source_infra_type, standardized_sector,
        standardized_category, is_flood_related
    )
) AS source
ON target.source_system = source.source_system
AND target.raw_category = source.raw_category
AND target.source_infra_type = source.source_infra_type
AND target.taxonomy_version = source.taxonomy_version
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT_IF(approval_status = 'APPROVED' AND is_active) AS active_approved_rows,
    COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
        'source_system', source_system,
        'raw_category', raw_category,
        'source_infra_type', source_infra_type,
        'taxonomy_version', taxonomy_version
    )) AS duplicate_natural_keys
FROM `02-silver`.config_project_category_mapping;

## 3. Project-status mappings

Grain: one source status per mapping version. Version `dpwh-status-2026-10-v1` seeds the five legitimate DPWH source statuses approved in issue #84. `For Procurement` maps to `Ongoing` and `ACTIVE`. `Not Yet Started` and `Terminated` map to `Inactive` and `INACTIVE`.

Blank, shifted, or malformed statuses, such as numbers or place names, get no rule. They stay `UNMAPPED` with `NULL` standardized values and remain visible in coverage output.

`Delayed` is not a supported target because the sources do not provide a reliable target completion date. Long-running and stalled are future derived measures, not status mappings.

In [0]:
CREATE TABLE IF NOT EXISTS `02-silver`.config_project_status_mapping (
    source_system STRING NOT NULL,
    source_status STRING NOT NULL,
    standardized_status STRING,
    status_group STRING,
    mapping_rule STRING,
    status_mapping_version STRING NOT NULL,
    approval_status STRING NOT NULL,
    approved_by STRING,
    approved_at TIMESTAMP,
    is_active BOOLEAN NOT NULL,
    source_reference STRING,
    notes STRING
)
USING DELTA
COMMENT 'Reviewed source-to-standard project status mappings';

MERGE INTO `02-silver`.config_project_status_mapping AS target
USING (
    SELECT
        source_system,
        source_status,
        standardized_status,
        status_group,
        'Exact trimmed DPWH status value' AS mapping_rule,
        'dpwh-status-2026-10-v1' AS status_mapping_version,
        'APPROVED' AS approval_status,
        'Buildabida' AS approved_by,
        TIMESTAMP '2026-10-09 00:00:00' AS approved_at,
        TRUE AS is_active,
        'Issue #84 approved decision and selected DPWH source profile' AS source_reference,
        'Blank, shifted, and malformed source statuses stay unmapped. Delayed is not used.' AS notes
    FROM VALUES
        ('dpwh_projects', 'Completed', 'Completed', 'FINISHED'),
        ('dpwh_projects', 'On-Going', 'Ongoing', 'ACTIVE'),
        ('dpwh_projects', 'For Procurement', 'Ongoing', 'ACTIVE'),
        ('dpwh_projects', 'Not Yet Started', 'Inactive', 'INACTIVE'),
        ('dpwh_projects', 'Terminated', 'Inactive', 'INACTIVE')
    AS seed(source_system, source_status, standardized_status, status_group)
) AS source
ON target.source_system = source.source_system
AND target.source_status = source.source_status
AND target.status_mapping_version = source.status_mapping_version
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT_IF(approval_status = 'APPROVED' AND is_active) AS active_approved_rows,
    COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
        'source_system', source_system,
        'source_status', source_status,
        'status_mapping_version', status_mapping_version
    )) AS duplicate_natural_keys
FROM `02-silver`.config_project_status_mapping;

## 4. MGB susceptibility mappings

Grain: one raw MGB code per source-contract and mapping version. Only the four documented codes are seeded. Blank values and `No rating` remain unmapped and visible in coverage output until a human decision is recorded.

In [0]:
CREATE TABLE IF NOT EXISTS `02-silver`.config_mgb_susceptibility_mapping (
    source_system STRING NOT NULL,
    raw_susceptibility_code STRING NOT NULL,
    standardized_level STRING,
    severity_rank INT,
    mapping_rule STRING,
    source_version STRING NOT NULL,
    mapping_version STRING NOT NULL,
    approval_status STRING NOT NULL,
    approved_by STRING,
    approved_at TIMESTAMP,
    is_active BOOLEAN NOT NULL,
    source_reference STRING,
    notes STRING
)
USING DELTA
COMMENT 'Versioned DENR MGB flood susceptibility code mappings';

MERGE INTO `02-silver`.config_mgb_susceptibility_mapping AS target
USING (
    SELECT *
    FROM VALUES
        ('DENR MGB flood susceptibility', 'LF', 'Low', 1, 'Official code-to-label contract', 'unversioned-official-code-contract', '2026-10-v1', 'APPROVED', TRUE, 'DENR MGB official susceptibility codes'),
        ('DENR MGB flood susceptibility', 'MF', 'Moderate', 2, 'Official code-to-label contract', 'unversioned-official-code-contract', '2026-10-v1', 'APPROVED', TRUE, 'DENR MGB official susceptibility codes'),
        ('DENR MGB flood susceptibility', 'HF', 'High', 3, 'Official code-to-label contract', 'unversioned-official-code-contract', '2026-10-v1', 'APPROVED', TRUE, 'DENR MGB official susceptibility codes'),
        ('DENR MGB flood susceptibility', 'VHF', 'Very High', 4, 'Official code-to-label contract', 'unversioned-official-code-contract', '2026-10-v1', 'APPROVED', TRUE, 'DENR MGB official susceptibility codes')
    AS seed(
        source_system, raw_susceptibility_code, standardized_level, severity_rank,
        mapping_rule, source_version, mapping_version, approval_status, is_active,
        source_reference
    )
) AS source
ON target.source_system = source.source_system
AND target.raw_susceptibility_code = source.raw_susceptibility_code
AND target.source_version = source.source_version
AND target.mapping_version = source.mapping_version
WHEN MATCHED AND NOT (
    target.standardized_level <=> source.standardized_level
    AND target.severity_rank <=> source.severity_rank
    AND target.mapping_rule <=> source.mapping_rule
    AND target.approval_status <=> source.approval_status
    AND target.is_active <=> source.is_active
    AND target.source_reference <=> source.source_reference
) THEN UPDATE SET
    standardized_level = source.standardized_level,
    severity_rank = source.severity_rank,
    mapping_rule = source.mapping_rule,
    approval_status = source.approval_status,
    is_active = source.is_active,
    source_reference = source.source_reference
WHEN NOT MATCHED THEN INSERT (
    source_system, raw_susceptibility_code, standardized_level, severity_rank,
    mapping_rule, source_version, mapping_version, approval_status, is_active,
    source_reference
) VALUES (
    source.source_system, source.raw_susceptibility_code, source.standardized_level,
    source.severity_rank, source.mapping_rule, source.source_version,
    source.mapping_version, source.approval_status, source.is_active,
    source.source_reference
);

In [0]:
SELECT
    raw_susceptibility_code,
    standardized_level,
    severity_rank,
    source_version,
    mapping_version,
    approval_status,
    is_active
FROM `02-silver`.config_mgb_susceptibility_mapping
ORDER BY severity_rank;

## 5. Manual geographic matches

Grain: one explicit source-record override per mapping version. This table is an exception path, not the default matching method. It starts empty and may remain empty. Every future row must identify its source record, approved target PSGC code, reason, and approver.

In [0]:
CREATE TABLE IF NOT EXISTS `02-silver`.config_manual_geographic_match (
    source_system STRING NOT NULL,
    record_type STRING NOT NULL,
    source_record_id STRING NOT NULL,
    raw_place_name STRING,
    raw_region_name STRING,
    raw_province_name STRING,
    raw_city_municipality_name STRING,
    canonical_psgc_code STRING,
    canonical_place_name STRING,
    override_reason STRING,
    mapping_version STRING NOT NULL,
    psgc_version STRING,
    approval_status STRING NOT NULL,
    approved_by STRING,
    approved_at TIMESTAMP,
    is_active BOOLEAN NOT NULL,
    source_reference STRING,
    notes STRING
)
USING DELTA
COMMENT 'Reviewed source-record-specific geographic exceptions';

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT_IF(approval_status = 'APPROVED' AND is_active) AS active_approved_rows,
    COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
        'source_system', source_system,
        'record_type', record_type,
        'source_record_id', source_record_id,
        'mapping_version', mapping_version
    )) AS duplicate_natural_keys
FROM `02-silver`.config_manual_geographic_match;

## Mapping coverage

The following query profiles only the distinct mapping domains needed for review. It does not transform or rewrite Bronze. The MGB table is grouped once in this notebook; the 2 GB source is not cached, sorted, or rescanned per code. Pending and unmapped values remain visible instead of being guessed.

In [0]:
WITH raw_dpwh_component_category AS (
    SELECT
        'dpwh_projects' AS source_system,
        '' AS raw_category,
        source_infra_type,
        COUNT(*) AS source_rows
    FROM (
        SELECT ARRAY_DISTINCT(FILTER(TRANSFORM(
            SPLIT(COALESCE(GET_JSON_OBJECT(source_record_json, '$.componentCategories'), ''), ','),
            token -> TRIM(token)
        ), token -> token <> '')) AS source_infra_types
        FROM `01-bronze`.dpwh_projects
    ) AS source
    LATERAL VIEW EXPLODE(source.source_infra_types) exploded AS source_infra_type
    GROUP BY source_infra_type
),
raw_project_category AS (
    SELECT * FROM raw_dpwh_component_category
    UNION ALL
    SELECT
        'flood_control_projects' AS source_system,
        COALESCE(TRIM(type_of_work), '') AS raw_category,
        COALESCE(TRIM(infra_type), '') AS source_infra_type,
        COUNT(*) AS source_rows
    FROM `01-bronze`.flood_control_projects
    GROUP BY
        COALESCE(TRIM(type_of_work), ''),
        COALESCE(TRIM(infra_type), '')
),
raw_project_status AS (
    SELECT
        'dpwh_projects' AS source_system,
        COALESCE(TRIM(status), '') AS source_status,
        COUNT(*) AS source_rows
    FROM `01-bronze`.dpwh_projects
    GROUP BY COALESCE(TRIM(status), '')
),
raw_place_alias AS (
    SELECT
        'dpwh_projects' AS source_system,
        COALESCE(TRIM(reported_region), '') AS raw_place_name,
        COUNT(*) AS source_rows
    FROM `01-bronze`.dpwh_projects
    GROUP BY COALESCE(TRIM(reported_region), '')
    UNION ALL
    SELECT
        'flood_control_projects' AS source_system,
        COALESCE(TRIM(reported_region), '') AS raw_place_name,
        COUNT(*) AS source_rows
    FROM `01-bronze`.flood_control_projects
    GROUP BY COALESCE(TRIM(reported_region), '')
),
raw_mgb AS (
    SELECT
        COALESCE(UPPER(TRIM(flood_susceptibility_code)), '') AS raw_code,
        COUNT(*) AS source_rows
    FROM `01-bronze`.flood_susceptibility
    GROUP BY COALESCE(UPPER(TRIM(flood_susceptibility_code)), '')
),
coverage AS (
    SELECT
        'project_category' AS mapping_domain,
        raw.source_system,
        CONCAT(raw.raw_category, ' | ', raw.source_infra_type) AS raw_value,
        raw.source_rows,
        mapping.approval_status,
        mapping.taxonomy_version AS mapping_version,
        mapping.is_active
    FROM raw_project_category AS raw
    LEFT JOIN `02-silver`.config_project_category_mapping AS mapping
        ON raw.source_system = mapping.source_system
        AND raw.raw_category = mapping.raw_category
        AND raw.source_infra_type = mapping.source_infra_type
        AND mapping.approval_status = 'APPROVED'
        AND mapping.is_active = TRUE
    UNION ALL
    SELECT
        'project_status' AS mapping_domain,
        raw.source_system,
        raw.source_status AS raw_value,
        raw.source_rows,
        mapping.approval_status,
        mapping.status_mapping_version AS mapping_version,
        mapping.is_active
    FROM raw_project_status AS raw
    LEFT JOIN `02-silver`.config_project_status_mapping AS mapping
        ON raw.source_system = mapping.source_system
        AND raw.source_status = mapping.source_status
        AND mapping.is_active
    UNION ALL
    SELECT
        'place_alias' AS mapping_domain,
        raw.source_system,
        raw.raw_place_name AS raw_value,
        raw.source_rows,
        mapping.approval_status,
        mapping.alias_version AS mapping_version,
        mapping.is_active
    FROM raw_place_alias AS raw
    LEFT JOIN `02-silver`.config_place_name_alias AS mapping
        ON raw.source_system = mapping.source_system
        AND raw.raw_place_name = mapping.raw_place_name
        AND mapping.raw_region_name = ''
        AND mapping.raw_province_name = ''
        AND mapping.raw_city_municipality_name = ''
        AND UPPER(TRIM(mapping.place_type)) = 'REGION'
        AND mapping.is_active
    UNION ALL
    SELECT
        'mgb_susceptibility' AS mapping_domain,
        'flood_susceptibility' AS source_system,
        raw.raw_code AS raw_value,
        raw.source_rows,
        mapping.approval_status,
        mapping.mapping_version,
        mapping.is_active
    FROM raw_mgb AS raw
    LEFT JOIN `02-silver`.config_mgb_susceptibility_mapping AS mapping
        ON raw.raw_code = mapping.raw_susceptibility_code
        AND mapping.is_active
)
SELECT
    mapping_domain,
    source_system,
    raw_value,
    source_rows,
    COALESCE(approval_status, 'UNMAPPED') AS review_state,
    mapping_version,
    CASE
        WHEN approval_status = 'APPROVED' AND is_active THEN 'READY'
        ELSE 'REVIEW_REQUIRED'
    END AS coverage_status
FROM coverage
ORDER BY mapping_domain, source_rows DESC, raw_value;

## Final summary

The next query reports table size, active approved rules, pending review rows, and visible mapping version. Empty tables are intentional until reviewers approve source-specific rules. Run the Silver configuration validator after this notebook.

In [0]:
SELECT 'config_place_name_alias' AS table_name, COUNT(*) AS row_count,
    COUNT_IF(approval_status = 'APPROVED' AND is_active) AS active_approved_rows,
    COUNT_IF(approval_status = 'PENDING_REVIEW') AS pending_rows,
    COALESCE(MAX(alias_version), 'NO_RULES_YET') AS current_version
FROM `02-silver`.config_place_name_alias
UNION ALL
SELECT 'config_project_category_mapping', COUNT(*),
    COUNT_IF(approval_status = 'APPROVED' AND is_active),
    COUNT_IF(approval_status = 'PENDING_REVIEW'),
    COALESCE(MAX(taxonomy_version), 'NO_RULES_YET')
FROM `02-silver`.config_project_category_mapping
UNION ALL
SELECT 'config_project_status_mapping', COUNT(*),
    COUNT_IF(approval_status = 'APPROVED' AND is_active),
    COUNT_IF(approval_status = 'PENDING_REVIEW'),
    COALESCE(MAX(status_mapping_version), 'NO_RULES_YET')
FROM `02-silver`.config_project_status_mapping
UNION ALL
SELECT 'config_mgb_susceptibility_mapping', COUNT(*),
    COUNT_IF(approval_status = 'APPROVED' AND is_active),
    COUNT_IF(approval_status = 'PENDING_REVIEW'),
    COALESCE(MAX(mapping_version), 'NO_RULES_YET')
FROM `02-silver`.config_mgb_susceptibility_mapping
UNION ALL
SELECT 'config_manual_geographic_match', COUNT(*),
    COUNT_IF(approval_status = 'APPROVED' AND is_active),
    COUNT_IF(approval_status = 'PENDING_REVIEW'),
    COALESCE(MAX(mapping_version), 'NO_RULES_YET')
FROM `02-silver`.config_manual_geographic_match
ORDER BY table_name;